# Résolution du VRPTW par Deep Learning
## Approche hybride : Imitation Learning + Beam Search + Cluster-then-Route

---

### Le problème — VRPTW

**n clients** à livrer avec une **fenêtre de temps** `[a_i, b_i]`, une **flotte de véhicules** de capacité Q, objectif : **minimiser la distance totale**.

Problème **NP-difficile** — les solveurs exacts (OR-Tools) deviennent impraticables au-delà de n ≈ 100.

---

### Notre contribution en 3 points

| | Contribution | Résultat |
|---|---|---|
| **1** | Attention Model entraîné par **imitation** d'OR-Tools | +6.5% gap vs optimal sur n=10 |
| **2** | **Décodeur contraint** : masquage dynamique des actions infaisables | **100% de faisabilité** sur toutes les tailles |
| **3** | **Cluster-then-Route** pour les grandes instances | n=1000 résolu en ~26s |

---

*Références : Kool et al. ICLR 2019 — Ye et al. AAAI 2024 (GLOP)*

---
## 1. Pistes explorées — ce que nous avons testé et abandonné

### Architecture commune : Attention Model (Kool et al., 2019)

Les deux approches utilisent la même architecture Transformer :

```
Clients (coords, TW, demande)  →  Encodeur Transformer  →  Embeddings h_i (N × 128)
                                                                     │
                                          Décodeur contextuel  ←─────┘
                                          score = 10·tanh(q·k / √d)
                                                     │
                                          Prochain client à visiter
```

---

### Piste 1 — REINFORCE (Kool 2019 original) ❌ Abandonné

Kool entraîne sans oracle : le modèle génère des solutions, reçoit `reward = -coût`.

```
Gradient REINFORCE : ∇L = -log π(a|s) × (reward - baseline)
Baseline           : rollout glouton du même modèle
```

**Problème rencontré sur VRPTW** : le sampling fait presque toujours *pire* que le greedy → avantage constamment négatif → gradient dit "réduis ces probabilités" sans jamais donner de signal positif.

Kool tourne sur **millions d'épisodes** (GPU, semaines). Nous avons ~25 000 épisodes max (CPU).

---

### Piste 2 — REINFORCE + EMA baseline + bonus entropie ❌ Insuffisant

Pour corriger le signal, nous avons implémenté :
- **Baseline EMA** : `b = 0.95·b + 0.05·coût_moyen` — compare à la moyenne récente, pas au greedy
- **Bonus entropie** : `loss -= 0.01·H(π)` — force l'exploration, évite la convergence prématurée

**Résultat** : coût descend de 577 → 512 en 200 epochs, mais reste très inférieur à l'imitation.

---

### Comparaison expérimentale

| Approche | Gap vs oracle (n=10, beam W=20) | Epochs | Verdict |
|---|---|---|---|
| REINFORCE (greedy baseline) | +35%+ | 200 | Signal toujours négatif |
| REINFORCE + EMA + entropie | +24.7% | 200 | Trop lent à converger |
| **Imitation Learning** | **+6.5%** | **100** | **Retenu** |

---
## 2. Solution retenue — Imitation Learning

### Pourquoi ça fonctionne

**Signal supervisé direct** : OR-Tools résout chaque instance de façon exacte. On extrait ses décisions pas à pas → dataset `(état, action_oracle)`. La loss est une simple cross-entropie.

```
Instance  →  OR-Tools  →  Décisions optimales
                                    │
              Dataset : (état_t, action_t*)  pour chaque étape t
                                    │
              Entraînement : minimiser  -log π(action_t* | état_t)
```

**Avantage sur VRPTW** : chaque décision est valide par construction — l'oracle ne viole jamais les TW ni la capacité. Le modèle n'apprend que de bons exemples.

### Pipeline complet

```
preprocess.py          →  Instance VRPTW (n, TW, Q, coords)
       │
oracle.py (OR-Tools)   →  Solution optimale pas-à-pas
       │
build_decision_dataset →  Dataset de décisions (état → action)
       │
train_small.py         →  Entraînement par imitation (cross-entropy)
       │
decoder.py             →  Décodeur CONTRAINT (masquage C1-C6)
       │
Beam Search W=20       →  Meilleure parmi 20 hypothèses parallèles
       │
DecodeResult           →  Routes faisables, coût, temps <1s
```

---
## 3. Démo live — Résolution n=10

**Greedy** (argmax à chaque étape) vs **Beam Search W=20** (20 hypothèses parallèles) vs **Oracle** OR-Tools

In [ ]:
import time, numpy as np, matplotlib.pyplot as plt
from preprocess import generate_instance
from decoder import decode_vrptw_beam_search, decode_vrptw_with_repair
from inference_decoder import load_trained_model, TorchModelScorer
from oracle import resoudre_instance

SEED, N = 10042, 10
instance = generate_instance(n=N, seed=SEED)
model    = load_trained_model('artifacts_train_small/best_model.pt')
scorer   = TorchModelScorer(model=model)

# ── Résolution ────────────────────────────────────────────────────────────────
t0 = time.time(); result_greedy = decode_vrptw_with_repair(instance, scorer=scorer); t_g = time.time()-t0
t0 = time.time(); result_beam   = decode_vrptw_beam_search(instance, scorer=scorer, beam_width=20); t_b = time.time()-t0
t0 = time.time(); sol_oracle    = resoudre_instance(instance, time_limit_s=10.0); t_o = time.time()-t0

oracle_cost = float(sol_oracle['cout']) if sol_oracle and sol_oracle['faisable'] else None
gap = lambda c, r: f'+{100*(c-r)/r:.1f}%' if r else 'N/A'

# ── Tableau résultats ─────────────────────────────────────────────────────────
print('=' * 60)
print(f'{"Methode":<26} {"Cout":>7}  {"Gap oracle":>10}  {"Temps":>8}')
print('-' * 60)
if oracle_cost:
    print(f'{"Oracle (OR-Tools)":<26} {oracle_cost:>7.1f}  {"—":>10}  {t_o:>7.1f}s')
print(f'{"AM + Greedy":<26} {result_greedy.total_cost:>7.1f}  {gap(result_greedy.total_cost, oracle_cost):>10}  {t_g*1000:>5.0f}ms')
print(f'{"AM + Beam Search W=20":<26} {result_beam.total_cost:>7.1f}  {gap(result_beam.total_cost, oracle_cost):>10}  {t_b*1000:>5.0f}ms')
print('=' * 60)

# ── Conversion tournées oracle → routes avec dépôt ────────────────────────────
def oracle_to_routes(sol):
    """Convertit tournees oracle (sans depot) en routes avec depot [0,...,0]."""
    if not sol:
        return []
    return [[0] + t + [0] for t in sol['tournees'] if t]

# ── Visualisation des routes ──────────────────────────────────────────────────
coords = instance['coords']
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
titles     = ['Oracle (OR-Tools)', 'AM + Greedy', 'AM + Beam Search W=20']
solutions  = [oracle_to_routes(sol_oracle), result_greedy.routes, result_beam.routes]
costs_list = [oracle_cost, result_greedy.total_cost, result_beam.total_cost]
colors     = plt.cm.tab10.colors

for ax, title, routes, cost in zip(axes, titles, solutions, costs_list):
    cost_str = f'{cost:.1f}' if cost else '?'
    ax.set_title(f'{title}\nCout : {cost_str}', fontsize=10, fontweight='bold')
    ax.scatter(coords[1:, 0], coords[1:, 1], c='steelblue', s=60, zorder=3)
    for i in range(1, N+1):
        ax.annotate(str(i), coords[i], fontsize=7, ha='center', va='bottom')
    ax.scatter(coords[0, 0], coords[0, 1], marker='*', c='red', s=200, zorder=5)
    for r_idx, route in enumerate(routes):
        if len(route) < 2:
            continue
        rx, ry = coords[route, 0], coords[route, 1]
        ax.plot(rx, ry, color=colors[r_idx % len(colors)], linewidth=1.8, alpha=0.8)
    ax.set_aspect('equal'); ax.axis('off')

plt.tight_layout()
plt.show()

---
## 4. Résultats — Benchmark 150 instances (5 tailles × 30 seeds)

**AM = Attention Model + Beam Search W=20 + réparation globale**

In [ ]:
import json, numpy as np, matplotlib.pyplot as plt
from scipy import stats

with open('../Benchmark/results/benchmark_summary.json') as f:
    rows = json.load(f)

sizes = [10, 20, 50, 100, 200]

# ── Tableau résumé ────────────────────────────────────────────────────────────
print(f'{"n":>5} | {"Fais.":>6} | {"Moy":>7} | {"Med":>7} | {"Std":>6} | {"Min":>6} | {"Max":>6} | {"IC 95%":>14}')
print('-' * 72)

stats_by_n = {}
for n in sizes:
    n_rows = [r for r in rows if r['n'] == n]
    am_ok  = sum(1 for r in n_rows if r['am_feasible'])
    gaps   = [r['am_gap_pct'] for r in n_rows if r['am_gap_pct'] is not None]
    g = np.array(gaps)
    ci = stats.t.interval(0.95, len(g)-1, loc=g.mean(), scale=stats.sem(g))
    stats_by_n[n] = g
    print(f'{n:>5} | {am_ok:>2}/{len(n_rows):<3} | {g.mean():>6.1f}% | {np.median(g):>6.1f}% | '
          f'{g.std():>5.1f}% | {g.min():>5.1f}% | {g.max():>5.1f}% | '
          f'[{ci[0]:.1f}%, {ci[1]:.1f}%]')

print('\n  AM = Attention Model + Beam Search W=20  |  100% faisabilite')

# ── Figure : boxplot + distribution ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot
ax = axes[0]
data = [stats_by_n[n] for n in sizes]
bp = ax.boxplot(data, labels=[str(n) for n in sizes], patch_artist=True, notch=False)
colors = plt.cm.Blues(np.linspace(0.4, 0.85, len(sizes)))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
ax.axhline(y=5, color='green', linestyle='--', linewidth=1.5, label='Kool 2019 (GPU, CVRP)')
ax.set_xlabel('Taille instance n')
ax.set_ylabel('Gap vs oracle (%)')
ax.set_title('Distribution des gaps — 30 instances par taille')
ax.legend()

# Histogramme global
ax2 = axes[1]
all_gaps = np.concatenate(list(stats_by_n.values()))
ax2.hist(all_gaps, bins=20, color='steelblue', alpha=0.8, edgecolor='white')
ax2.axvline(all_gaps.mean(), color='red', linestyle='--', linewidth=2, label=f'Moyenne = {all_gaps.mean():.1f}%')
ax2.axvline(np.median(all_gaps), color='orange', linestyle='--', linewidth=2, label=f'Mediane = {np.median(all_gaps):.1f}%')
ax2.set_xlabel('Gap vs oracle (%)')
ax2.set_ylabel('Nombre d\'instances')
ax2.set_title('Distribution globale des gaps (150 instances)')
ax2.legend()

plt.tight_layout()
plt.show()

---
## 4b. Étude statistique — Comparaison des modèles (n10 vs mixte)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
import time

from preprocess import generate_instance
from decoder import decode_vrptw_beam_search
from inference_decoder import load_trained_model, TorchModelScorer
from oracle import resoudre_instance

# ── Chargement des modèles ────────────────────────────────────────────────────
model_n10    = load_trained_model('artifacts_train_small/best_model.pt')
model_mixed  = load_trained_model('artifacts_train_small/best_model_mixed.pt')
scorer_n10   = TorchModelScorer(model=model_n10)
scorer_mixed = TorchModelScorer(model=model_mixed)

# ── Benchmark sur 30 seeds, toutes tailles ───────────────────────────────────
SIZES      = [10, 20, 50, 100, 200]
SEEDS      = list(range(2000, 2030))   # 30 seeds dédiées à l'étude stat
BEAM_WIDTH = 10
ORACLE_MAX = 20

gaps_n10   = {n: [] for n in SIZES}
gaps_mixed = {n: [] for n in SIZES}
times_n10  = {n: [] for n in SIZES}
times_mx   = {n: [] for n in SIZES}

oracle_costs = {n: [] for n in SIZES}

print("Calcul en cours (30 seeds x 5 tailles)...")
for n in SIZES:
    for seed in SEEDS:
        inst = generate_instance(n=n, seed=seed)

        t0 = time.time()
        r10 = decode_vrptw_beam_search(inst, scorer=scorer_n10, beam_width=BEAM_WIDTH)
        times_n10[n].append(time.time() - t0)

        t0 = time.time()
        rmx = decode_vrptw_beam_search(inst, scorer=scorer_mixed, beam_width=BEAM_WIDTH)
        times_mx[n].append(time.time() - t0)

        if n <= ORACLE_MAX:
            sol = resoudre_instance(inst, time_limit_s=10.0)
            if sol and sol['faisable']:
                ref = float(sol['cout'])
                oracle_costs[n].append(ref)
                gaps_n10[n].append(100*(r10.total_cost - ref)/ref)
                gaps_mixed[n].append(100*(rmx.total_cost - ref)/ref)

    print(f"  n={n} OK")

# ── Tableau statistique complet ───────────────────────────────────────────────
print('\n' + '='*90)
print(f'{"n":>5} | {"Modele":>8} | {"Moy":>7} | {"Med":>7} | {"Std":>6} | {"Min":>6} | {"Max":>6} | {"IC 95%":>16} | {"t(ms)":>6}')
print('-'*90)

for n in [10, 20]:
    for label, gaps, times in [('n10', gaps_n10[n], times_n10[n]),
                                ('mixed', gaps_mixed[n], times_mx[n])]:
        g  = np.array(gaps)
        ci = stats.t.interval(0.95, len(g)-1, loc=g.mean(), scale=stats.sem(g))
        tm = np.mean(times)*1000
        print(f'{n:>5} | {label:>8} | {g.mean():>6.1f}% | {np.median(g):>6.1f}% | '
              f'{g.std():>5.1f}% | {g.min():>5.1f}% | {g.max():>5.1f}% | '
              f'[{ci[0]:.1f}%,{ci[1]:.1f}%] | {tm:>5.0f}')
    print('-'*90)

# Coût relatif pour n=50,100,200 (pas d'oracle → delta mixed vs n10)
for n in [50, 100, 200]:
    c10 = np.array([decode_vrptw_beam_search(generate_instance(n=n, seed=s),
                    scorer=scorer_n10, beam_width=BEAM_WIDTH).total_cost for s in SEEDS[:10]])
    cmx = np.array([decode_vrptw_beam_search(generate_instance(n=n, seed=s),
                    scorer=scorer_mixed, beam_width=BEAM_WIDTH).total_cost for s in SEEDS[:10]])
    delta = 100*(cmx - c10)/c10
    ci = stats.t.interval(0.95, len(delta)-1, loc=delta.mean(), scale=stats.sem(delta))
    print(f'{n:>5} | {"n10→mix":>8} | {delta.mean():>+6.1f}% | {np.median(delta):>+6.1f}% | '
          f'{delta.std():>5.1f}% | {delta.min():>+5.1f}% | {delta.max():>+5.1f}% | '
          f'[{ci[0]:+.1f}%,{ci[1]:+.1f}%] |   —')

print('='*90)
print('  IC = Intervalle de Confiance a 95% (Student t)')

# ── Figure : comparaison n10 vs mixed ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Comparaison statistique : Modele n10 vs Modele mixte (n10+n20+n50)', fontweight='bold')

# Boxplot comparatif n=10 et n=20 (avec oracle)
ax = axes[0]
data = [gaps_n10[10], gaps_mixed[10], gaps_n10[20], gaps_mixed[20]]
labels = ['n10\n(n=10)', 'mixed\n(n=10)', 'n10\n(n=20)', 'mixed\n(n=20)']
bp = ax.boxplot(data, labels=labels, patch_artist=True)
colors_box = ['#5B9BD5', '#2E75B6', '#ED7D31', '#C55A11']
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
ax.set_ylabel('Gap vs oracle (%)')
ax.set_title('Gap vs oracle — n=10 et n=20 (30 seeds)')
ax.axhline(0, color='gray', linestyle=':', linewidth=0.8)

# Temps d'exécution par taille et modèle
ax2 = axes[1]
x = np.arange(len(SIZES))
w = 0.35
t10_ms = [np.mean(times_n10[n])*1000 for n in SIZES]
tmx_ms = [np.mean(times_mx[n])*1000 for n in SIZES]
ax2.bar(x - w/2, t10_ms, w, label='Modele n10',    color='#5B9BD5', alpha=0.85)
ax2.bar(x + w/2, tmx_ms, w, label='Modele mixed',  color='#ED7D31', alpha=0.85)
ax2.set_xticks(x); ax2.set_xticklabels([str(n) for n in SIZES])
ax2.set_xlabel('Taille instance n')
ax2.set_ylabel('Temps moyen (ms)')
ax2.set_title('Temps de résolution par taille (Beam W=10)')
ax2.legend()

plt.tight_layout()
plt.show()

---
## 5. Grandes instances — Cluster-then-Route (n=1000)

**Problème** : notre Transformer en O(N²) ne passe pas à l'échelle directement.  
**Solution** : décomposition géographique inspirée de GLOP (Ye et al., AAAI 2024).

---
## 6. Bilan et perspectives

### Récapitulatif

| Méthode | Gap n=10 | Faisabilité | Temps | Scalable |
|---|---|---|---|---|
| Oracle (OR-Tools) | 0% | 100% | 5–120s | Non (n>100) |
| AM Greedy | ~14% | 100% | <100ms | Oui |
| **AM + Beam W=20** | **+6.5%** | **100%** | **<1s** | **Oui** |
| AM REINFORCE (200 ep.) | ~25% | 100% | <1s | Oui |
| **Cluster-then-Route** | **+10% (n=1000)** | **100%** | **~26s** | **Oui** |

### Ce qui nous rapproche de Kool 2019
- Même architecture Transformer + clipping 10·tanh
- Beam Search à l'inférence (même idée que le sampling ×1280 de Kool)
- Masquage dynamique des contraintes à chaque étape

### Perspectives immédiates
1. **Entraînement multi-taille** (n=10+20+50) → Cluster-then-Route passe de +10% à ~+6%
2. **Fine-tuning REINFORCE** depuis le checkpoint imitation → meilleur des deux mondes
3. **Sparse Attention** → résolution directe n=1000 sans clustering

---
## 9. Grandes instances — Approche Cluster-then-Route

### Pourquoi le modèle seul ne suffit pas sur n=1000

Notre Transformer calcule une attention **O(N²)** entre tous les nœuds. Sur n=1000 :
- Matrice d'attention : 1001 × 1001 ≈ 4 Mo par couche
- Beam search W=20 : à chaque étape, 20 faisceaux × ~1000 clients = décodage très lent
- Résultat : +18.6% de gap vs OR-Tools (lui-même non optimal sur n=1000)

### Solution : Cluster-then-Route

Inspiré de l'approche industrielle (GLOP 2022, ORTEC) : on **décompose** le problème en sous-problèmes que notre modèle sait déjà résoudre.

```
n=1000 clients
      │
  K-Means géographique
  (K-Means++ maison, sans sklearn)
      │
  20 clusters de ~50 clients
  ┌──────┬──────┬──────┬─────┐
  │Cl. 1 │Cl. 2 │ ...  │Cl.20│  ← résolution indépendante
  └──────┴──────┴──────┴─────┘
      │       │           │
  AM+Beam  AM+Beam    AM+Beam    ← notre modèle existant, inchangé
  W=10     W=10        W=10
      │
  Recombinaison des routes
      │
  Réparation globale (si clients manquants aux frontières)
      │
  Solution finale n=1000
```

### Analogie avec les métaheuristiques

| Métaheuristique | Notre approche |
|---|---|
| Décomposition / sous-problèmes | Clusters géographiques |
| Solveur local | AM + Beam Search sur chaque cluster |
| Recombinaison | Merge des routes + réparation globale |
| Intensification | Beam width W élevé par cluster |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import time

from preprocess import generate_instance
from large_instance_solver import cluster_clients, solve_large_instance
from inference_decoder import load_trained_model, TorchModelScorer

# ── Génération de l'instance ──────────────────────────────────────────────────
N_DEMO = 200   # réduit pour affichage rapide (même logique que n=1000)
SEED   = 42
instance = generate_instance(n=N_DEMO, seed=SEED)

model  = load_trained_model('artifacts_train_small/best_model.pt')
scorer = TorchModelScorer(model=model)

# ── Clustering ────────────────────────────────────────────────────────────────
CLUSTER_SIZE = 40
clusters = cluster_clients(instance, cluster_size=CLUSTER_SIZE, seed=0)
k = len(clusters)

# Assignation label pour chaque client
labels = np.zeros(N_DEMO + 1, dtype=int)
for c_idx, cluster in enumerate(clusters):
    for node in cluster:
        labels[node] = c_idx

coords = instance["coords"]   # shape (N+1, 2)

# ── Résolution Cluster-then-Route ─────────────────────────────────────────────
t0 = time.time()
result = solve_large_instance(instance, scorer, cluster_size=CLUSTER_SIZE, beam_width=10, verbose=False)
t_ctr  = time.time() - t0

# ── Figure ────────────────────────────────────────────────────────────────────
cmap = plt.cm.get_cmap('tab20', k)
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle(f'Cluster-then-Route — n={N_DEMO} clients, {k} clusters de ~{CLUSTER_SIZE} nœuds',
             fontsize=14, fontweight='bold')

# ---------- Panneau gauche : clusters ----------
ax = axes[0]
ax.set_title('Étape 1 — Clustering K-Means géographique', fontsize=12)
for c_idx in range(k):
    mask = [i for i in range(1, N_DEMO + 1) if labels[i] == c_idx]
    cx = coords[mask, 0]
    cy = coords[mask, 1]
    ax.scatter(cx, cy, color=cmap(c_idx), s=20, alpha=0.8)
    # Centroïde
    ax.scatter(cx.mean(), cy.mean(), marker='x', color=cmap(c_idx),
               s=80, linewidths=2, zorder=5)

# Dépôt
ax.scatter(coords[0, 0], coords[0, 1], marker='*', color='black', s=300,
           zorder=10, label='Dépôt')
ax.legend(fontsize=9)
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_aspect('equal')

# ---------- Panneau droit : routes ----------
ax2 = axes[1]
ax2.set_title(f'Étape 2 — Routes (coût total : {result.total_cost:.0f})', fontsize=12)

# Fond clients par cluster (pâle)
for c_idx in range(k):
    mask = [i for i in range(1, N_DEMO + 1) if labels[i] == c_idx]
    ax2.scatter(coords[mask, 0], coords[mask, 1],
                color=cmap(c_idx), s=15, alpha=0.3)

# Tracer les routes
for route in result.routes:
    if len(route) < 2:
        continue
    rx = coords[route, 0]
    ry = coords[route, 1]
    ax2.plot(rx, ry, color='steelblue', linewidth=0.8, alpha=0.6)

# Dépôt
ax2.scatter(coords[0, 0], coords[0, 1], marker='*', color='black', s=300, zorder=10)

n_routes  = len(result.routes)
n_served  = len(result.served_clients)
n_unserved = len(result.unserved_clients)

ax2.set_xlabel('x'); ax2.set_ylabel('y')
ax2.set_aspect('equal')

# Légende stats
stats = (f'Routes : {n_routes}\n'
         f'Clients servis : {n_served}/{N_DEMO}\n'
         f'Non servis : {n_unserved}\n'
         f'Temps : {t_ctr:.1f}s')
ax2.text(0.02, 0.98, stats, transform=ax2.transAxes,
         fontsize=9, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()

print(f'\n✓ Cluster-then-Route terminé en {t_ctr:.1f}s')
print(f'  {k} clusters × ~{CLUSTER_SIZE} clients → {n_routes} routes')
print(f'  Clients servis : {n_served}/{N_DEMO}  |  Coût : {result.total_cost:.1f}')
